In [ ]:
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans, DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

# Dim reduction
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import umap # <== NOT a scikit-learn library, but follows the interface

import matplotlib.pyplot as plt
import seaborn as sns
import hvplot.pandas

# Unsupervised models in scikit-learn: clustering and dimentionality reduction

An earlier lecturer presented the scikit-learn library and its overall design. In learning that design, the students were introduced to supervised learning. Like the earlier lecture, the goal of this lecture is not to teach students all details of clustering techniques. This lecture remains an introduction to the API used for relevant tasks in data science.

### Cluster trades into common categories
Our task is to build a table containing common attributes of all US stocks: volume, closing price, number of transactions, average size of trades, etc. Note that "average/mean size of trades" needs to be calculated from a large file containing all trades done for a day. 

Once we have enrichsed end-of-day data for each stock with mean_size calculation, we will start the process of building clusters.

Note that in a real-world scenario, _lots_ more features could be added as input to the clustering algorithm.

#### Read EOD data

In [ ]:
eod_data_df = pd.read_csv('../../datasets/market_data/2025-09-10_OCHLV.csv.gz').set_index('ticker')
eod_data_df

#### Read market data file and calc mean trade size

Please note that 2025-09-10_trades_small.csv.gz is a large file and will be made available to students via canvas. If it has not been made available, please contact the lecturer for the file.

In [ ]:
%%time
tick_df = pd.read_csv('/Users/shahbaz/Dropbox/uchicago_lecturer/data/processed/trades_2025-09-10_small_sorted.csv.gz')
#tick_df.sip_timestamp = pd.to_datetime(tick_df.sip_timestamp, unit='ns')
tick_df.head()

In [ ]:
mean_execution_size_df = tick_df[['ticker', 'size']].groupby('ticker').mean()
del tick_df

In [ ]:
mean_execution_size_df.head()

#### Create a security master with EOD data and mean_size column

In [ ]:
sec_master_df = pd.concat([eod_data_df, mean_execution_size_df], axis=1)
sec_master_df.rename({'size':'mean_size'}, axis=1, inplace=True)
sec_master_df.dropna(inplace=True)
sec_master_df.drop(columns=['open', 'high', 'low', 'window_start'], inplace=True) #drop the columns we are not using

In [ ]:
sec_master_df

**A note about this data being unrealistic**

The code below demonstrates unsupervised learning using scikit-learn. In a more realistic scenario, many other features could be considered. For example, stock prices are not stationry (in time-series terms), perhaps price changes could be included. 

Additionally, a better defined reason for clustering could suggest additional features. For example, if this clustering is being done for a smart order router, then time related featuers, such as seconds since the open, seconds since a halt, days until earnings, etc. could be included.

#### Create clusters

In [ ]:
cluster_input_cols = sec_master_df.columns.tolist()
cluster_input_cols

In [ ]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('kmeans', KMeans(n_clusters=5, random_state=42, n_init='auto'))
])

In [ ]:
model = pipeline.named_steps['kmeans']

In [ ]:
model.n_clusters

The method `fit_transform` returns the distance between cluster centers (centroid) and each record. Not useful for data analysis

In [ ]:
transformed_distances = pipeline.fit_transform(sec_master_df)
transformed_distances.shape

In [ ]:
column_names = [f"distance_to_cluster_{i}" for i in range(model.n_clusters)]
df_distances = pd.DataFrame(transformed_distances, columns=column_names)
df_distances.head()

The method `fit_predict` returns cluster labels for each input record

In [ ]:
labels = pipeline.fit_predict(sec_master_df)
labels, labels.shape

The model object also provides labels, number of clusters, etc.

In [ ]:
model.labels_, model.labels_.shape

Annotate the "security master" with cluster labels

In [ ]:
sec_master_df['cluster_label'] = model.labels_.astype(pd.CategoricalDtype) # annotating types can often help with performance and memory usage
sec_master_df.head()

Is the distribution of data points very skewed?

In [ ]:
sec_master_df.cluster_label.value_counts()

In [ ]:
sns.catplot(
    data=pd.melt(sec_master_df, id_vars = 'cluster_label'), 
    x='variable', y='value', col='cluster_label', kind='strip', col_wrap=5, sharey=False
)

Could it be that we should not have picked 5 clusters?

#### Pick the right cluster size

Let's try different cluster sizes and measure their performance.

Notice that if we use a Pipeline inside the loop, we are computing the StandardScaler multiple times for the same data. This wastes cpu cycles! Let's break apart the two tasks and bring them together in a pipeline later

```python
for i in range(2, 10 + 1):
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('kmeans', KMeans(n_clusters=5, random_state=42, n_init='auto'))
    ])
    ...
```

In [ ]:
sec_master_scaled_df = StandardScaler().fit_transform(sec_master_df)

In [ ]:
%%time
inertia_scores = list()
silhouette_scores = list()

for i in range (2, 10 +1):
    kmeans = KMeans(n_clusters=i, random_state=42, n_init='auto')
    cluster_labels = kmeans.fit_predict(sec_master_scaled_df)
    
    inertia_scores.append(kmeans.inertia_) # <= Needed for the "elbow" method

    silhouette_scores.append(silhouette_score(sec_master_scaled_df, cluster_labels) ) # <= Needed for the "silhouette" score method

**Elbow method**

This is perhaps the most standard way to determine the correct size for a cluster. Data scientists visually review the part of the chart which has the sharpest "kink" or "elbow". In the chart below, that value is 4 or 5

The elbow method calculates the Within-Cluster Sum of Squared (WCSS) distances. 

In [ ]:
pd.DataFrame({'k':range(2,10+1), 'inertia':inertia_scores}).plot.line(x='k', y='inertia')

**Silhouette method**

Another way to determine the correct cluster size is to measure the mean distance to all other points in the _same_ cluster and the mean distance to every point in the _nearest neighboring_ cluster. A ratio of these numbers is called the "silhouette score." [1] Higher values are better.

> "The best value is 1 and the worst value is -1. Values near 0 indicate overlapping clusters. Negative values generally indicate that a sample has been assigned to the wrong cluster, as a different cluster is more similar.
" - Scikit learn docs

Note that 'cohesion' and 'separation' are related measures and often useful measuring the quality of clusters, such as text topic models.

[1] https://scikit-learn.org/stable/modules/generated/sklearn.metrics.silhouette_score.html

In [ ]:
pd.DataFrame({'k':range(2,10+1), 'silhouette':silhouette_scores}).plot.line(x='k', y='silhouette')

#### Pick a better cluster size
Given this analysis, let's pick the cluster size to be 4

In [ ]:
%%time

CLUSTER_SIZE = 4

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('kmeans', KMeans(n_clusters=CLUSTER_SIZE, random_state=42, n_init='auto'))
])

labels = pipeline.fit_predict(sec_master_df)
labels, labels.shape

model = pipeline.named_steps['kmeans']

sec_master_df['cluster_label'] = model.labels_.astype(pd.CategoricalDtype) # annotating types can often help with performance and memory usage

In [ ]:
model.get_params()

#### Investigate clusters

Books and internet examples (and LLMs!) often build for loops to iterate through various clusters and build cluster specific charts. In the example below, we use Pandas' `melt` function to created "faceted" charts.

In [ ]:
sec_master_df

Unpivot the dataframe (see Pandas lecture on pivot/melt)

In [ ]:
pd.melt(filtered_on_cluster_df)

In [ ]:
pd.melt(sec_master_df[sec_master_df.columns], id_vars = 'cluster_label')

In [ ]:
pd.melt(sec_master_df, id_vars = 'cluster_label')

In [ ]:
sns.catplot(
    data=pd.melt(sec_master_df, id_vars = 'cluster_label'), 
    x='cluster_label', y='value', col='variable', kind='bar', col_wrap=4, sharey=False
)

In [ ]:
sns.catplot(
    data=pd.melt(sec_master_df, id_vars = 'cluster_label'), 
    x='variable', y='value', col='cluster_label', kind='strip', col_wrap=4, sharey=False
)

#### For good measure: try DBSCAN

In [ ]:
%%time

pipeline_dbs = Pipeline([
    ('scaler', StandardScaler()),
    ('dbscan', DBSCAN())
])

pipeline_dbs.fit(sec_master_df)

model_dbs = pipeline_dbs.named_steps['dbscan']

sec_master_df['cluster_label'] = model_dbs.labels_.astype(pd.CategoricalDtype) # annotating types can often help with performance and memory usage

In [ ]:
model_dbs.get_params()

Cluster -1 is noise

In [ ]:
n_clusters_dbs = len(set([int(l) for l in model_dbs.labels_ if l != -1]))
n_clusters_dbs

In [ ]:
sns.catplot(
    data=pd.melt(sec_master_df[sec_master_df.cluster_label != -1], id_vars = 'cluster_label'), 
    x='cluster_label', y='value', col='variable', kind='bar', col_wrap=4, sharey=False
)

In [ ]:
sns.catplot(
    data=pd.melt(sec_master_df[sec_master_df.cluster_label != -1], id_vars = 'cluster_label'), 
    x='variable', y='value', col='cluster_label', kind='strip', col_wrap=n_clusters_dbs, sharey=False
)

#### Reduce dimensions and plot

PCA

In [ ]:
%%time
X_reduced = PCA(n_components=2).fit_transform(sec_master_df)
X_reduced.shape

pd.concat([
    pd.DataFrame(X_reduced, columns=['x', 'y']), 
    pd.DataFrame({'label':pd.Series(model.labels_, dtype='category')})]
    , axis=1)\
    .plot.scatter(x='x', y='y', c='label', colormap='viridis')

TSNE

In [ ]:
%%time
X_reduced = TSNE(n_components=2, learning_rate='auto', init='random', perplexity=3).fit_transform(sec_master_df)
X_reduced.shape

pd.concat([
    pd.DataFrame(X_reduced, columns=['x', 'y']), 
    pd.DataFrame({'label':pd.Series(model.labels_, dtype='category')})]
    , axis=1)\
    .plot.scatter(x='x', y='y', c='label', colormap='viridis')

UMAP

In [ ]:
%%time
X_reduced = umap.UMAP().fit_transform(sec_master_df) # <== Thsi is not even from scikit-learn!
X_reduced.shape

pd.concat([
    pd.DataFrame(X_reduced, columns=['x', 'y']), 
    pd.DataFrame({'label':pd.Series(model.labels_, dtype='category')})]
    , axis=1)\
    .plot.scatter(x='x', y='y', c='label', colormap='viridis')